In [1]:
from matplotlib.pyplot import figure, rc
import numpy as np
import matplotlib.pyplot as plt
import time
from ipywidgets import interact
import ipywidgets as widgets
from IPython.display import display


def logistic_growth_function(r, x):
    return r*x*(1-x)
    # 
    # N' = r*N*(K-N)
    # Entdimensionalisierung: neue Variable x=N/K 
    # 
def sine_growth_function (r,x):
    return r*np.sin(np.pi*x)/4
    
def iterate_logistic_growth_function(n_iterations, r, x_0):
    if type(x_0) == float or type(x_0) == np.longdouble:
        values = np.zeros(n_iterations, dtype=np.longdouble)
    else:
        values = np.zeros((n_iterations,x_0.shape[-1]), dtype=np.longdouble)
    values[0] = x_0
    for i in range(1, n_iterations):
        values[i] = logistic_growth_function(r, values[i-1])
    return values

def iterate_sine_growth_function(n_iterations, r, x_0):
    if type(x_0) == float or type(x_0) == np.longdouble:
        values = np.zeros(n_iterations, dtype=np.longdouble)
    else:
        values = np.zeros((n_iterations,x_0.shape[-1]), dtype=np.longdouble)
    values[0] = x_0
    for i in range(1, n_iterations):
        values[i] = sine_growth_function(r, values[i-1])
    return values


def plot_cobweb(r=0.1, x0=0.2, tmax=100, growth_function=logistic_growth_function, n_times_function_applied=1):
    x = np.linspace(0,1,500)
    fig = plt.figure(figsize=(15,8), dpi=100)
    ax=fig.add_subplot(111)
    def final_map(r,x):
        result = growth_function(r,x)
        for i in range(n_times_function_applied-1):
            result = growth_function(r, result)
        return result
    ax.plot(x, final_map(r,x), c='#444444', lw=2)
    ax.plot(x,x, c='#444444', lw=2)

    px, py = np.empty((2, tmax+1, 2))
    px[0] = x0
    py[0] = 0
    for n in range(1, tmax, 2):
        px[n] = px[n-1]
        py[n] = final_map(r, px[n-1])
        px[n+1] = py[n]
        py[n+1] = py[n]

    ax.plot(px, py, c="b", alpha=0.7)
    ax.set_xlim((0,1))
    ax.set_ylim((0,1))
    ax.set_xlabel("Anfangspopulationsgröße x")
    ax.set_ylabel("Endwert Populationsgröße $rx(1-x)$")
    ax.set_title('$x_0 = {:.1}, r = {:.7}$'.format(x0*1.,r*1.))

def plot_orbit_diagram(r_values=(3.0, 3.86),r_value_distance=0.0001, x0=0.5, n_iterations_to_plot_from=100, n_iterations_to_plot_until=200, labelsize=0.001, figsize=(15,8), dpi=100, discrete_map=iterate_logistic_growth_function, color="tab:blue", ax=None):
    all_r_values = np.arange(*r_values,r_value_distance)
    
    scatter_points = np.zeros((all_r_values.shape[-1], -n_iterations_to_plot_from+n_iterations_to_plot_until))
    for i,r in enumerate(all_r_values):
        scatter_points[i] = discrete_map(n_iterations_to_plot_until, r, x0)[n_iterations_to_plot_from:n_iterations_to_plot_until]
    
    fig, ax = plt.subplots(figsize=figsize, dpi=dpi)
    fig.subplots_adjust(left=0.06,right=0.99,bottom=0.1,top=0.95)
    fig.canvas.header_visible = False
    fig.canvas.toolbar_visible = False
    ax.scatter(np.repeat(all_r_values, -n_iterations_to_plot_from+n_iterations_to_plot_until), scatter_points.flatten(), s=labelsize, marker=".", c=color)
    ax.set_xlim(r_values)
    ax.set_ylim(0,1)
    ax.set_xlabel("r")
    ax.set_ylabel("Werte von x für $t \\in ({:},{:})$".format(n_iterations_to_plot_from, n_iterations_to_plot_until))
    ax.set_title('$x_0 = {:.1}, r \\in ({:.9},{:.9})$'.format(x0*1.0,r_values[0]*1.,r_values[1]*1.))
    fig.canvas.draw()
    
    return ax

def plot_trajectory_old_function(r=3.0, x0=0.5, n_iterations=100, figsize=(20,10), dpi=100):
    plt.figure(figsize=figsize, dpi=dpi)
    x = np.arange(1, n_iterations+1)
    y = iterate_logistic_growth_function(n_iterations, r, np.longdouble(x0))
    
    plt.plot(x[:n_iterations], y[:n_iterations])
    plt.xlabel("Zeitschritte t")
    plt.ylabel("Populationsgröße x")
    plt.ylim(0,1)
    plt.xlim(0,n_iterations)
    plt.title('$x_0 = {:.1}, r = {:.4}$'.format(x0*1.,r*1.))
%matplotlib widget

In [8]:
n_iterations = 100
x0 = x0_init = 0.5
figsize = (9,4.5)
dpi = 100
r_init = 0

with plt.ioff():
    fig, ax = plt.subplots(figsize=(figsize))
fig.subplots_adjust(left=0.06,right=0.99,bottom=0.1,top=0.98)
fig.canvas.header_visible = False
fig.canvas.toolbar_visible = False

x = np.arange(0,n_iterations)
y = iterate_logistic_growth_function(n_iterations, r_init, np.longdouble(x0_init))

plot, = ax.plot(x, y)
ax.set_xlabel("Zeitschritte t")
ax.set_ylabel("Populationsgröße x")
ax.set_ylim(0,1)
ax.set_xlim(0,n_iterations)


def plot_trajectory(r):
    y = iterate_logistic_growth_function(n_iterations, r, np.longdouble(x0))
    #plt.plot(x[:n_iterations], y[:n_iterations])
    plot.set_ydata(y)
    fig.canvas.draw()
r_slider = widgets.FloatSlider(value=0.0,min=0.0, max=4.0, step=0.001,description="r",disabled=False,continuous_update=True,orientation="horizontal",readout=True, readout_format=".3f",layout=widgets.Layout(width='1000px'))
#x0_slider = widgets.FloatSlider(value=0.5,min=0.0, max=1.0, step=0.001,description="Anfangswert",disabled=False,continuous_update=True,orientation="vertical",readout=True, readout_format=".3f",layout=widgets.Layout(height='500px'))
w = widgets.interactive(plot_trajectory, r=r_slider)
ui = widgets.VBox([r_slider, fig.canvas], layout=widgets.Layout(width='100%',display='flex', align_items='center'))
display(ui)

In [9]:
tmax=300
n_times_function_applied=1
rc_init = 0.0
x0 = x0_init=0.5
xc_init = np.linspace(0,1,tmax)
with plt.ioff():
    figc,axc=plt.subplots(figsize=(4,4), dpi=100)
growth_function=logistic_growth_function

def final_map(r,x):
    result = growth_function(r,x)
    for i in range(n_times_function_applied-1):
        result = growth_function(r, result)
    return result
# quadratic
quadratic_plot, = axc.plot(xc_init, final_map(rc_init,xc_init), c='#444444', lw=2)
# linear
linear_plot, = axc.plot(xc_init,xc_init, c='#444444', lw=2)

# plot of the steps
px, py = np.empty((2, tmax+1, 2))
px[0] = x0_init
py[0] = 0
for n in range(1, tmax, 2):
    px[n] = px[n-1]
    py[n] = final_map(rc_init, px[n-1])
    px[n+1] = py[n]
    py[n+1] = py[n]
steps_plot_1, steps_plot_2 = axc.plot(px, py, c="b", alpha=0.7)

axc.set_xlim((0,1))
axc.set_ylim((0,1))
axc.set_xlabel("Anfangspopulationsgröße x")
axc.set_ylabel("Endwert Populationsgröße $rx(1-x)$")

    

def plot_cobweb(r):
    quadratic_plot.set_ydata(final_map(r,xc_init))
    axc.set_title('$x_0 = {:.3}, r = {:.7}$'.format(x0*1.,r*1.))
    px, py = np.empty((2, tmax+1, 2))
    px[0] = x0
    py[0] = 0
    for n in range(1, tmax, 2):
        px[n] = px[n-1]
        py[n] = final_map(r, px[n-1])
        px[n+1] = py[n]
        py[n+1] = py[n]
    steps_plot_1.set_xdata(px)
    steps_plot_1.set_ydata(py)
    steps_plot_2.set_xdata(px)
    steps_plot_2.set_ydata(py)
    
    figc.canvas.draw()

figc.subplots_adjust(left=0.15,right=0.9,bottom=0.15,top=0.9)
figc.canvas.header_visible = False
figc.canvas.toolbar_visible = False


rc_slider = widgets.FloatSlider(value=rc_init,min=0.0, max=4.0, step=0.001,description="r",disabled=False,continuous_update=True,orientation="horizontal",readout=True, readout_format=".3f",layout=widgets.Layout(width='500px', align_items="initial"))
x0c_slider = widgets.FloatSlider(value=x0_init,min=0.0, max=1.0, step=0.001,description="Anfangswert",disabled=False,continuous_update=True,orientation="vertical",readout=True, readout_format=".3f",layout=widgets.Layout(height='400px'))
wc = widgets.interactive(plot_cobweb, r=rc_slider)

ui2 = widgets.VBox([rc_slider, figc.canvas], layout=widgets.Layout(height='100%',display='flex', align_items='center'))

figc.canvas.draw()
display(ui2)